# 1. MediaPipe Pose Data Cleaning
Steps:
1. Read the MediaPipe CSV files (one file per participant).
2. Select the landmarks of interest.
3. Build a global long dataframe (without Z, without `participant`).
4. Invert the Y-axis (top of the image = large y).
5. Interpolate missing coordinates.
6. Convert from long format to wide format.
7. Save the cleaned dataframe as a CSV file.

## 1.1 Importations et configuration

In [1]:
import sys
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent.parent

from config import (
    DATA_DIR,
    LM,
    LANDMARKS_OF_INTEREST,
    ID_COLS,
)

print(f"Project dir : {PROJECT_DIR}")
print(f"Data dir    : {DATA_DIR}")

Project dir : C:\Users\factory\Desktop\pons.lea
Data dir    : C:\Users\factory\Desktop\pons.lea\data


## 1.2 Identifying Participating Files

We look for files named `landmarks_*_POSE.csv` in `data/movement/landmarks/P` and `data/movement/landmarks/SZ`.

In [2]:
def list_participant_files(data_dir):
    """Returns the list of MediaPipe Pose CSV files in data/movement/landmarks/P and data/movement/landmarks/SZ."""
    data_dir = Path(data_dir)

    p_files = sorted(
        glob.glob(str(data_dir / "movement"  / "landmarks" / "P" / "landmarks_*_POSE.csv"))
    )

    sz_files = sorted(
        glob.glob(str(data_dir / "movement" / "landmarks" / "SZ" / "landmarks_*_POSE.csv"))
    )

    return p_files, sz_files, p_files + sz_files


def parse_participant_from_filename(filepath):
    """
    Extract the participant ID from the filename.
    Expected format: landmarks_P1_POSE.csv or landmarks_SZ12_POSE.csv
    """
    name = Path(filepath).stem
    match = re.match(r"landmarks_(P|SZ)(\d+)_POSE", name)
    if match is None:
        raise ValueError(f"Invalid filename : {name}")
    group, num = match.group(1), match.group(2)
    return {
        "group": group,
        "participant_id": f"{group}{num}",
        "filepath": filepath,
    }


# Creating the list of participants
p_files, sz_files, all_files = list_participant_files(DATA_DIR)
participant_rows = [parse_participant_from_filename(f) for f in all_files]
participants_df = pd.DataFrame(participant_rows)

print(f"Group P  : {len(p_files)} files")
print(f"Group SZ : {len(sz_files)} files")
print(f"Total    : {len(all_files)} files")
display(participants_df.head())

Group P  : 20 files
Group SZ : 20 files
Total    : 40 files


,group,participant_id,filepath
0,P,P10,C:\Users\factory\Desktop\pons.lea\data\movemen...
1,P,P11,C:\Users\factory\Desktop\pons.lea\data\movemen...
2,P,P12,C:\Users\factory\Desktop\pons.lea\data\movemen...
3,P,P13,C:\Users\factory\Desktop\pons.lea\data\movemen...
4,P,P14,C:\Users\factory\Desktop\pons.lea\data\movemen...


## 1.3 Building the long data frame

For each participant, we read their CSV file and keep only the landmarks
of interest and the `x` and `y` columns (the `z` coordinate is ignored).
We then concatenate everything into a single long data frame.

In [3]:
def read_long_csv(filepath):
    """
   Reads a MediaPipe CSV file in long format.

    Expected input columns: frame, index, x, y (z is ignored).
    Returns: frame, landmark, x, y.
    """
    df = pd.read_csv(filepath)
    required = ["frame", "index", "x", "y"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Columns missing in {filepath} : {missing}")
    df["landmark"] = df["index"].map(LM)
    return df[["frame", "landmark", "x", "y"]]


def build_long_dataframe(participant_rows, landmarks=LANDMARKS_OF_INTEREST):
    """
    Builds a long-format DataFrame from the CSV files of all participants.
    Only keeps the landmarks of interest and the columns x, y
    (no z column, no `participant` column).
    """
    frames = []
    for row in participant_rows:
        df = read_long_csv(row["filepath"])
        df = df[df["landmark"].isin(landmarks)].copy()
        df["group"] = row["group"]
        df["participant_id"] = row["participant_id"]
        frames.append(df)
    df_long = pd.concat(frames, ignore_index=True)
    return df_long[["group", "participant_id", "frame", "landmark", "x", "y"]]


df_signals_long = build_long_dataframe(participant_rows)

print(f"Shape long          : {df_signals_long.shape}")
print(f"Participants        : {df_signals_long['participant_id'].nunique()}")
print(f"Conserved Landmarks : {sorted(df_signals_long['landmark'].unique())}")
display(df_signals_long.head())

Shape long          : (178174, 6)
Participants        : 40
Conserved Landmarks : ['left_shoulder', 'left_wrist', 'nose', 'right_shoulder', 'right_wrist']


,group,participant_id,frame,landmark,x,y
0,P,P10,0,nose,0.516045,0.161254
1,P,P10,0,left_shoulder,0.573700,0.273409
2,P,P10,0,right_shoulder,0.464896,0.268849
3,P,P10,0,left_wrist,0.570096,0.538465
4,P,P10,0,right_wrist,0.452303,0.550361


## 1.4 Inverting the Y-Axis

MediaPipe places the origin at the top of the image (y=0 at the top, y=1 at the bottom).
We invert this to obtain a natural vertical axis: large y = top.

In [4]:
def invert_y_axis(df_long):
    """Invert the Y axis : y_new = 1 - y_old."""
    df = df_long.copy()
    df["y"] = 1 - df["y"]
    return df


df_signals_long = invert_y_axis(df_signals_long)
display(df_signals_long.head())

,group,participant_id,frame,landmark,x,y
0,P,P10,0,nose,0.516045,0.838746
1,P,P10,0,left_shoulder,0.573700,0.726591
2,P,P10,0,right_shoulder,0.464896,0.731151
3,P,P10,0,left_wrist,0.570096,0.461535
4,P,P10,0,right_wrist,0.452303,0.449639


## 1.5 Interpolation of Missing Values

For each (participant, landmark) pair, missing frames were linearly interpolated if:

the proportion of missing values was ≤ max_missing_prop,
the longest sequence of consecutive NaNs was ≤ max_gap.

Otherwise, the entire landmark signal was set to NaN for that participant.

**Report**: This cell was initially run using a maximum gap of 90 frames and a maximum proportion of 10% missing values. For all signals that did not meet these criteria, I inspected the corresponding videos to identify the periods during which the landmark was no longer detected. None of these cases corresponded to moments when the landmark actually disappeared from the video. Instead, they mainly occurred when, for example, the arms were positioned behind the back and remained still. I therefore relaxed the criteria so that these segments, which corresponded to stationary and otherwise clearly identifiable body parts, could be linearly interpolated. I then visually checked the resulting signals against the videos to ensure that the interpolated trajectories were consistent with the observed movements.

In [5]:
def count_complete_landmark_signals(df_long):
    """Count the number of participants with a complete signal for each landmark."""
    rows = []

    for landmark, sub in df_long.groupby("landmark", sort=False):
        complete = (
            sub.groupby("participant_id")[["x", "y"]]
            .agg(lambda s: s.notna().all())
            .all(axis=1)
        )

        rows.append({
            "landmark": landmark,
            "n_total": sub["participant_id"].nunique(),
            "n_complete": int(complete.sum()),
        })

    return pd.DataFrame(rows).sort_values("landmark").reset_index(drop=True)


def _max_consecutive_nans(mask):
    """Find the longest sequence of consecutive True values in a boolean array."""
    longest = current = 0
    for is_nan in mask:
        current = current + 1 if is_nan else 0
        longest = max(longest, current)
    return longest


def interpolate_coordinates(df_long, max_missing_prop=0.30, max_gap=250):
    """
    Linearly interpolate missing (x, y) coordinates.

    For each (participant, landmark):
        - if there are no missing values -> keep the signal unchanged;
        - if missing values meet the interpolation criteria -> interpolate;
        - otherwise -> set the entire landmark signal to NaN.

    The frame range is calculated per participant to detect missing frames
    at the beginning or end of individual landmark trajectories.

    Returns
    -------
    df_interp : pd.DataFrame
        Long-format DataFrame containing the processed coordinates.
    report : pd.DataFrame
        Processing report for each (participant, landmark) pair.
    """
    pid_to_group = (
        df_long.drop_duplicates("participant_id")
        .set_index("participant_id")["group"]
    )

    pid_to_frame_range = (
        df_long.groupby("participant_id")["frame"]
        .agg(["min", "max"])
    )

    pieces = []
    report_rows = []

    for (pid, landmark), sub in df_long.groupby(
        ["participant_id", "landmark"], sort=False
    ):
        sub = sub.sort_values("frame").copy()

        f_min = pid_to_frame_range.loc[pid, "min"]
        f_max = pid_to_frame_range.loc[pid, "max"]
        all_frames = np.arange(f_min, f_max + 1)

        sub = (
            sub.set_index("frame")
            .reindex(all_frames)
            .rename_axis("frame")
            .reset_index()
        )

        sub["participant_id"] = pid
        sub["landmark"] = landmark
        sub["group"] = pid_to_group[pid]

        nan_mask = sub["x"].isna() | sub["y"].isna()
        n_missing = int(nan_mask.sum())
        n_total = len(sub)
        prop_missing = nan_mask.mean()
        max_gap_len = _max_consecutive_nans(nan_mask.to_numpy())
        missing_frames = sub.loc[nan_mask, "frame"].tolist()

        if prop_missing == 0:
            status = "complete"

        elif prop_missing <= max_missing_prop and max_gap_len <= max_gap:
            sub[["x", "y"]] = (
                sub[["x", "y"]]
                .interpolate(method="linear", limit_direction="both")
            )
            status = "interpolated"

        else:
            sub[["x", "y"]] = np.nan
            status = "rejected"

        report_rows.append({
            "participant_id": pid,
            "landmark": landmark,
            "status": status,
            "n_frames_total": n_total,
            "n_missing": n_missing,
            "prop_missing": round(prop_missing, 4),
            "max_gap": max_gap_len,
            "missing_frames": missing_frames,
        })

        pieces.append(sub)

    df_interp = pd.concat(pieces, ignore_index=True)
    df_interp = df_interp[
        ["group", "participant_id", "frame", "landmark", "x", "y"]
    ]

    report = pd.DataFrame(report_rows)
    return df_interp, report


# Before-and-after comparison
print("Before interpolation:")
display(count_complete_landmark_signals(df_signals_long))

df_signals_long, interp_report = interpolate_coordinates(
    df_signals_long,
    max_missing_prop=0.30,
    max_gap=300,
)

print("\nAfter interpolation:")
display(count_complete_landmark_signals(df_signals_long))


# Processing summary
print("\n=== Processing summary ===")
display(interp_report.groupby("status").size().rename("count"))


# Details of interpolated signals
interpolated = interp_report[interp_report["status"] == "interpolated"]

print(f"\n=== Interpolated signals ({len(interpolated)}) ===")

if not interpolated.empty:
    display_df = interpolated.copy()
    display_df["missing_frames"] = display_df["missing_frames"].apply(
        lambda lst: f"{lst[:10]}... ({len(lst)} total)"
        if len(lst) > 10 else str(lst)
    )

    display(display_df[
        ["participant_id", "landmark", "n_missing",
         "prop_missing", "max_gap", "missing_frames"]
    ])

else:
    print("None.")


# Details of rejected signals
rejected = interp_report[interp_report["status"] == "rejected"]

print(f"\n=== Rejected signals ({len(rejected)}) ===")

if not rejected.empty:
    display_df = rejected.copy()
    display_df["missing_frames"] = display_df["missing_frames"].apply(
        lambda lst: f"{lst[:10]}... ({len(lst)} total)"
        if len(lst) > 10 else str(lst)
    )

    display(display_df[
        ["participant_id", "landmark", "n_missing",
         "prop_missing", "max_gap", "missing_frames"]
    ])

else:
    print("None.")


# Verification
print("\n=== Verification: are all rejected signals 100% NaN? ===")

if rejected.empty:
    print("No rejected signals.")

else:
    all_nan_check = []

    for _, row in rejected.iterrows():
        pid = row["participant_id"]
        landmark = row["landmark"]

        mask = (
            (df_signals_long["participant_id"] == pid)
            & (df_signals_long["landmark"] == landmark)
        )

        sub = df_signals_long.loc[mask]

        x_all_nan = sub["x"].isna().all()
        y_all_nan = sub["y"].isna().all()

        all_nan_check.append({
            "participant_id": pid,
            "landmark": landmark,
            "x_all_nan": x_all_nan,
            "y_all_nan": y_all_nan,
            "ok": x_all_nan and y_all_nan,
        })

    check_df = pd.DataFrame(all_nan_check)

    if check_df["ok"].all():
        print(
            f"OK: all {len(check_df)} rejected signals "
            "are indeed 100% NaN."
        )
    else:
        print("PROBLEM: some rejected signals are not 100% NaN:")
        display(check_df[~check_df["ok"]])

Before interpolation:


,landmark,n_total,n_complete
0,left_shoulder,40,40
1,left_wrist,39,39
2,nose,40,40
3,right_shoulder,40,40
4,right_wrist,39,39



After interpolation:


,landmark,n_total,n_complete
0,left_shoulder,40,40
1,left_wrist,39,39
2,nose,40,40
3,right_shoulder,40,40
4,right_wrist,39,39



=== Processing summary ===


status
complete        186
interpolated     12
Name: count, dtype: int64


=== Interpolated signals (12) ===


,participant_id,landmark,n_missing,prop_missing,max_gap,missing_frames
3,P10,left_wrist,261,0.2852,171,"[441, 442, 443, 444, 445, 446, 447, 448, 449, ..."
4,P10,right_wrist,145,0.1585,74,"[441, 442, 443, 444, 445, 446, 447, 448, 449, ..."
34,P16,right_wrist,4,0.0043,4,"[0, 1, 2, 3]"
43,P18,left_wrist,10,0.0109,5,"[134, 135, 136, 137, 138, 318, 389, 390, 391, ..."
44,P18,right_wrist,93,0.1016,66,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]... (93 total)"
64,P2,right_wrist,27,0.0300,19,"[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]... (2..."
108,SZ11,left_wrist,88,0.0975,88,"[774, 775, 776, 777, 778, 779, 780, 781, 782, ..."
133,SZ16,left_wrist,268,0.2971,233,"[564, 567, 568, 570, 571, 572, 573, 574, 575, ..."
153,SZ1,left_wrist,16,0.0173,16,"[624, 625, 626, 627, 628, 629, 630, 631, 632, ..."
154,SZ1,right_wrist,231,0.2503,57,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]... (231 total)"



=== Rejected signals (0) ===
None.

=== Verification: are all rejected signals 100% NaN? ===
No rejected signals.


## 1.6 Switching to wide format

One row = one participant and one frame. The columns become
`nose_x`, `nose_y`, `left_shoulder_x`, etc. No Z, no
redundant `participant` column.

In [6]:
def long_to_wide(df_long):
    """
    Rotate the long dataframe (x, y per landmark) into a wide format.
    One row = (participant, frame). No Z, no `participant`.
    """
    df_wide = df_long.pivot_table(
        index=["group", "participant_id", "frame"],
        columns="landmark",
        values=["x", "y"],
    )
    # Flatten the MultiIndex of columns : ("x", "nose") -> "nose_x".
    df_wide.columns = [f"{landmark}_{coord}" for coord, landmark in df_wide.columns]
    return (
        df_wide.reset_index()
        .sort_values(["participant_id", "frame"])
        .reset_index(drop=True)
    )


df_signals_wide = long_to_wide(df_signals_long)

print(f"Shape wide : {df_signals_wide.shape}")
display(df_signals_wide.head())

Shape wide : (36238, 13)


,group,participant_id,frame,left_shoulder_x,left_wrist_x,nose_x,right_shoulder_x,right_wrist_x,left_shoulder_y,left_wrist_y,nose_y,right_shoulder_y,right_wrist_y
0,P,P1,0,0.551109,0.513249,0.494273,0.437366,0.451267,0.764519,0.506094,0.884955,0.771039,0.499536
1,P,P1,1,0.551934,0.513923,0.495771,0.437479,0.451267,0.764174,0.505475,0.885404,0.770886,0.499538
2,P,P1,2,0.553424,0.514413,0.496511,0.438607,0.451295,0.764080,0.504557,0.885955,0.771016,0.500116
3,P,P1,3,0.555237,0.514497,0.498662,0.440048,0.451630,0.763992,0.504500,0.886327,0.771771,0.500956
4,P,P1,4,0.556226,0.514459,0.500877,0.441361,0.451765,0.764004,0.504875,0.887334,0.773063,0.502561


## 1.7 Saving the Cleaned DataFrame

Output: `data/movement/01_clean_signals.csv`. This file will be
the input for Notebook 2 (filtering).

In [7]:
MOVEMENT_DIR = DATA_DIR / "movement"

MOVEMENT_DIR.mkdir(parents=True, exist_ok=True)

out_path = MOVEMENT_DIR / "01_clean_signals.csv"

df_signals_wide.to_csv(out_path, index=False)

print(f"Saved cleaned signals -> {out_path}")

display(df_signals_wide.head())

Saved cleaned signals -> C:\Users\factory\Desktop\pons.lea\data\movement\01_clean_signals.csv


,group,participant_id,frame,left_shoulder_x,left_wrist_x,nose_x,right_shoulder_x,right_wrist_x,left_shoulder_y,left_wrist_y,nose_y,right_shoulder_y,right_wrist_y
0,P,P1,0,0.551109,0.513249,0.494273,0.437366,0.451267,0.764519,0.506094,0.884955,0.771039,0.499536
1,P,P1,1,0.551934,0.513923,0.495771,0.437479,0.451267,0.764174,0.505475,0.885404,0.770886,0.499538
2,P,P1,2,0.553424,0.514413,0.496511,0.438607,0.451295,0.764080,0.504557,0.885955,0.771016,0.500116
3,P,P1,3,0.555237,0.514497,0.498662,0.440048,0.451630,0.763992,0.504500,0.886327,0.771771,0.500956
4,P,P1,4,0.556226,0.514459,0.500877,0.441361,0.451765,0.764004,0.504875,0.887334,0.773063,0.502561
